In [8]:
import ast
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

BLOCK_SIZE = 64
ANNOTATIONS_PATH = "../data/annotations.csv"
EMBEDDINGS_DIR = Path("../jepa2/embeddings")

# Data preparation

We first process the annotations to obtain the *action* (row in annotations) corresponding to each *block* (group of 64 frames)

In [9]:
df = pd.read_csv(ANNOTATIONS_PATH, usecols=["video_id", "relevant", "start_frame", "stop_frame"])

# We transform the unit of time from frames to blocks
df["start_block"] = df["start_frame"] // BLOCK_SIZE
df["stop_block"] = df["stop_frame"] // BLOCK_SIZE

# We create copies of the rows to have one for each block in which the action is happening
# Example: if we have an action from block 10 to 20, we will create 10 copies,
#           for block 10, 11, 12, ..., 20
rows = []
for _, row in df.iterrows():
    for block in range(row["start_block"], row["stop_block"] + 1):
        rows.append({"video_id": row["video_id"], "block": block, "relevant": row["relevant"]})

# Expanded is the df from annotations but with copied rows
expanded = pd.DataFrame(rows)

# To each block embedding we tag the relevance depending on the rows of annotations.csv corresponding to that video and block
# For each (video_id, block): no row → False, one row → its value, multiple rows → any True

# expanded.groupby.any groups the relevance labels for a (video, block) pair and
# saves true if any of the actions in that block is true
# block_labels is a mapping of (video_id, block_idx) -> relevance
block_labels = (
    expanded.groupby(["video_id", "block"])["relevant"]
    .any()
)

print(f"Unique videos in annotations: {df['video_id'].nunique()}")
print(expanded.head())

Unique videos in annotations: 495
  video_id  block  relevant
0   P01_01      0     False
1   P01_01      1     False
2   P01_01      2     False
3   P01_01      3     False
4   P01_01      4      True


We then create two arrays, one for the block embeddings and one for the block labels

In [10]:
# Load embeddings and assign block-level labels
all_embeddings = []
all_labels = []

for pkl_path in sorted(EMBEDDINGS_DIR.glob("*.pkl")):
    video_id = pkl_path.stem

    with pkl_path.open("rb") as f:
        payload = pickle.load(f)

    embeddings = payload["embeddings"] if isinstance(payload, dict) else payload
    num_blocks = len(embeddings)

    labels = [bool(block_labels.get((video_id, block), False)) for block in range(num_blocks)]

    all_embeddings.append(embeddings)
    all_labels.extend(labels)

embeddings_tensor = np.concatenate(all_embeddings, axis=0)
labels_array = np.array(all_labels, dtype=bool)

print(f"Total blocks: {len(labels_array)}  relevant: {labels_array.sum()}  irrelevant: {(~labels_array).sum()}")

Total blocks: 8980  relevant: 5042  irrelevant: 3938


## Train test split

In [11]:
import torch
from sklearn.model_selection import train_test_split

X = torch.tensor(embeddings_tensor, dtype=torch.float32)
y = torch.tensor(labels_array, dtype=torch.float32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {len(X_train)} blocks  Test: {len(X_test)} blocks")
print(f"Train positive rate: {y_train.mean():.2%}  Test positive rate: {y_test.mean():.2%}")

Train: 7184 blocks  Test: 1796 blocks
Train positive rate: 56.15%  Test positive rate: 56.12%


# Model definition

In [12]:
import torch.nn as nn

input_dim = X.shape[1]

model = nn.Sequential(
    nn.Linear(input_dim, 512),
    nn.LayerNorm(512),
    nn.GELU(),
    nn.Dropout(0.25),
    nn.Linear(512, 256),
    nn.LayerNorm(256),
    nn.GELU(),
    nn.Dropout(0.25),
    nn.Linear(256, 128),
    nn.LayerNorm(128),
    nn.GELU(),
    nn.Dropout(0.25),
    nn.Linear(128, 1),
)

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Input dim: {input_dim}  Device: {device}")

Input dim: 1024  Device: mps


# Training

In [13]:
from torch.utils.data import DataLoader, TensorDataset

EPOCHS = 20
BATCH_SIZE = 256
LR = 1e-3

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(X_batch).squeeze(-1), y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_batch)

    print(f"epoch={epoch:02d}  loss={total_loss / len(y_train):.4f}")

epoch=01  loss=0.6661
epoch=02  loss=0.6202
epoch=03  loss=0.5804
epoch=04  loss=0.5539
epoch=05  loss=0.5486
epoch=06  loss=0.5202
epoch=07  loss=0.5294
epoch=08  loss=0.4918
epoch=09  loss=0.4703
epoch=10  loss=0.4642
epoch=11  loss=0.4648
epoch=12  loss=0.4545
epoch=13  loss=0.4223
epoch=14  loss=0.4101
epoch=15  loss=0.3953
epoch=16  loss=0.4426
epoch=17  loss=0.3707
epoch=18  loss=0.3481
epoch=19  loss=0.3565
epoch=20  loss=0.3575


# Evaluation

In [15]:
model.eval()
with torch.no_grad():
    logits = model(X_test.to(device)).squeeze(-1).cpu()
    preds = (torch.sigmoid(logits) >= 0.5)

correct = (preds == y_test.bool()).sum().item()
tp = ((preds == 1) & (y_test == 1)).sum().item()
tn = ((preds == 0) & (y_test == 0)).sum().item()
fp = ((preds == 1) & (y_test == 0)).sum().item()
fn = ((preds == 0) & (y_test == 1)).sum().item()

precision = tp / max(1, tp + fp)
recall = tp / max(1, tp + fn)
f1 = 2 * precision * recall / max(1e-12, precision + recall)

print(f"accuracy:  {correct / len(y_test):.3f}")
print(f"precision: {precision:.3f}")
print(f"recall:    {recall:.3f}")
print(f"f1:        {f1:.3f}")
print()
print(f"TP (correctly predicted relevant):    {tp}")
print(f"TN (correctly predicted irrelevant):  {tn}")
print(f"FP (predicted relevant, was not):     {fp}")
print(f"FN (predicted irrelevant, was not):   {fn}")

accuracy:  0.766
precision: 0.809
recall:    0.764
f1:        0.786

TP (correctly predicted relevant):    770
TN (correctly predicted irrelevant):  606
FP (predicted relevant, was not):     182
FN (predicted irrelevant, was not):   238
